# 공통 유틸 모듈 동작 검증

연도별 원본 노트북을 수정하지 않고 `pipeline_common`, `text_match`, `llm_refine`의 핵심 동작을 합성 데이터로 검증한다. 외부 LLM API는 호출하지 않는다.

In [1]:
import json
import sys
from functools import partial
from pathlib import Path
from types import SimpleNamespace
from unittest.mock import Mock

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.features.llm_refine import (  # noqa: E402
    call_llm_once,
    clean_sentence,
    needs_llm_rerun,
    numbers_preserved,
)
from src.features.pipeline_common import (  # noqa: E402
    assign_labels,
    build_subtotal_qa,
    calculate_budget_changes,
    classify_row,
    clean_text,
    normalize_budget_type,
    show_table1_around,
    to_numeric_budget,
)
from src.features.text_match import (  # noqa: E402
    check_pattern,
    check_pattern_broad,
    dedup_label,
    extract_via_regex,
    find_near_duplicates,
    normalize_for_match,
)

print(f"프로젝트 루트: {PROJECT_ROOT}")
print("공통 모듈 import 완료")

프로젝트 루트: /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha
공통 모듈 import 완료


## 1. 파이프라인 공통 함수 검증

In [2]:
df_raw = pd.DataFrame(
    {
        "지역": ["서울"] * 5,
        "원본행": [1, 2, 3, 4, 5],
        "세부사업명": [
            "Ⅰ. 공통사업",
            "1. 함께 일하고 함께 돌보는 사회(공통)",
            "아이 돌봄 지원",
            "노인복지관 운영 지원",
            "세부사업명",
        ],
        "사업분류재정구분": [pd.NA, pd.NA, "공 통", " 자 체 ", pd.NA],
        "2024년 예산": ["1,500", "1,200", "1,000", "-", pd.NA],
        "2023년 예산": ["1,400", "1,100", "900", "-", pd.NA],
        "주요내용": [
            pd.NA,
            pd.NA,
            "• 지원대상: 아동\n지원내용: 돌봄 제공",
            "운영  비용 지원",
            pd.NA,
        ],
    }
)

df_raw["사업행구분"] = df_raw["세부사업명"].apply(classify_row)
df_labeled = df_raw.groupby("지역", group_keys=False).apply(assign_labels)
df_labeled["지역"] = df_raw.loc[df_labeled.index, "지역"]
df_labeled["예산_num"] = to_numeric_budget(df_labeled["2024년 예산"], zero_tokens=("-",))
subtotal_qa = build_subtotal_qa(df_labeled, budget_col="예산_num")
df_leaf = df_labeled[df_labeled["사업행구분"] == "세부사업"].copy()
df_leaf["예산_num"] = to_numeric_budget(df_leaf["2024년 예산"], zero_tokens=("-",))
previous_budget = to_numeric_budget(df_leaf["2023년 예산"], zero_tokens=("-",))
budget_changes = calculate_budget_changes(df_leaf["예산_num"], previous_budget)
df_leaf[["당해예산", "전년도예산", "증감액", "증감율"]] = budget_changes
df_leaf["세부사업명"] = clean_text(df_leaf["세부사업명"])
df_leaf["주요내용"] = clean_text(df_leaf["주요내용"], strip_leading_bullet=True)
df_leaf["사업분류재정구분"] = normalize_budget_type(df_leaf["사업분류재정구분"])

assert df_raw["사업행구분"].tolist() == [
    "대분류_소계",
    "중분류_소계",
    "세부사업",
    "세부사업",
    "헤더반복",
]
assert df_leaf["예산_num"].tolist() == [1000, 0]
assert df_leaf["증감액"].tolist() == [100, 0]
assert df_leaf.iloc[0]["증감율"] == 11.1
assert pd.isna(df_leaf.iloc[1]["증감율"])
assert df_leaf["사업분류재정구분"].tolist() == ["공통", "자체"]
assert df_leaf.iloc[0]["주요내용"] == "지원대상: 아동 지원내용: 돌봄 제공"
assert df_leaf["대분류"].eq("Ⅰ. 공통사업").all()
assert subtotal_qa.iloc[0]["원본_소계값"] == 1200
assert subtotal_qa.iloc[0]["leaf_합계"] == 1000
assert subtotal_qa.iloc[0]["차이"] == -200
assert subtotal_qa.iloc[0]["결과"] == "불일치"

display(
    df_leaf[
        [
            "지역",
            "대분류",
            "중분류",
            "세부사업명",
            "사업분류재정구분",
            "당해예산",
            "전년도예산",
            "증감액",
            "증감율",
            "주요내용",
        ]
    ]
)
display(subtotal_qa)
print("pipeline_common 검증 통과")

,지역,대분류,중분류,세부사업명,사업분류재정구분,당해예산,전년도예산,증감액,증감율,주요내용
2,서울,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),아이 돌봄 지원,공통,1000,900,100,11.1,지원대상: 아동 지원내용: 돌봄 제공
3,서울,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),노인복지관 운영 지원,자체,0,0,0,<NA>,운영 비용 지원


,지역,대분류,중분류,원본_소계값,leaf_합계,차이,QA_병합상태,결과
0,서울,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),1200,1000,-200,양쪽존재,불일치


pipeline_common 검증 통과


In [3]:
df_table1 = pd.DataFrame(
    {
        0: ["사업1", "사업2", "사업3", "사업4", "사업5"],
        1: [None] * 5,
        2: ["공통", "공통", "자체", "자체", "공통"],
        3: [10, 20, 30, 40, 50],
    }
)

table_view = show_table1_around(
    df_table1,
    center_excel_row=3,
    window=1,
    label="합성 원본 대조",
)
assert table_view["세부사업명"].tolist() == ["사업2", "사업3", "사업4"]
assert table_view["예산"].tolist() == [20, 30, 40]
display(table_view)

--- 합성 원본 대조 (Table 1 엑셀행 2~4) ---


,세부사업명,공통/자체,예산
1,사업2,공통,20
2,사업3,자체,30
3,사업4,자체,40


## 2. 텍스트 매칭·라벨 추출 검증

In [4]:
match_df = pd.DataFrame(
    {
        "지역": ["서울", "서울", "서울", "부산"],
        "중분류": ["돌봄"] * 4,
        "세부사업명": [
            "노인복지관 운영 지원",
            "노인복지관 운영지원 확대",
            "청년 월세 지원",
            "노인복지관 운영지원 확대",
        ],
        "당해예산": [10, 20, 30, 40],
        "주요내용": ["A", "B", "C", "D"],
    }
)

near_duplicates = find_near_duplicates(match_df, threshold=80)
assert len(near_duplicates) == 1
assert near_duplicates.iloc[0]["지역"] == "서울"
assert normalize_for_match("노인 복지관-운영(지원)") == "노인복지관운영지원"

labeled_text = dedup_label("• 지원대상: 지원대상： 서울시민 (지원내용) 돌봄 제공")
assert labeled_text == "지원대상 : 서울시민 지원내용 : 돌봄 제공"
assert check_pattern(labeled_text) == "일치"
assert check_pattern_broad("사업대상 서울시민 사업내용 돌봄 제공") == "일치"
assert extract_via_regex("사업대상: 서울시민 사업내용: 돌봄 제공") == {
    "지원대상": "서울시민",
    "지원내용": "돌봄 제공",
}

display(near_duplicates)
print("text_match 검증 통과")

,지역,중분류,세부사업명1,당해예산1,주요내용1,세부사업명2,당해예산2,주요내용2,유사도
0,서울,돌봄,노인복지관 운영 지원,10,A,노인복지관 운영지원 확대,20,B,83.333333


text_match 검증 통과


## 3. LLM 보존형 교정 검증(모의 응답)

In [5]:
raw_response = json.dumps({"정제문장": "월 30만 원씩 3개월 지원"}, ensure_ascii=False)
response = SimpleNamespace(choices=[SimpleNamespace(message=SimpleNamespace(content=raw_response))])
create_mock = Mock(return_value=response)
mock_client = SimpleNamespace(chat=SimpleNamespace(completions=SimpleNamespace(create=create_mock)))
llm_cfg = {
    "upstage": {"model": "solar-pro3", "temperature": 0},
    "prompt": {"template": "사업명: {name}\n주요내용: {content}"},
    "response_schema": {"name": "refined_text", "schema": {"type": "object"}},
}
call_once = partial(call_llm_once, client=mock_client, llm_config=llm_cfg)

cleaned = clean_sentence(
    "지원 사업",
    "월 30만원씩 3개월 지원",
    call_once=call_once,
)
assert cleaned == "월 30만 원씩 3개월 지원"
assert numbers_preserved("월 30만원씩 3개월 지원", cleaned)
assert needs_llm_rerun("(지원대상) 서울시민")
assert not needs_llm_rerun("지원대상: 서울시민")
assert create_mock.call_count == 1

print(f"모의 LLM 결과: {cleaned}")
print("llm_refine 검증 통과")

모의 LLM 결과: 월 30만 원씩 3개월 지원
llm_refine 검증 통과


## 검증 결론

세 공통 모듈의 핵심 동작을 합성 데이터와 모의 LLM 응답으로 확인한다. 이 검증은 실제 연도별 데이터의 리팩터링 전후 QA 비교를 대신하지 않으며, 노트북 적용 단계에서 별도의 회귀 검증이 필요하다.

In [6]:
verification_summary = pd.Series(
    {
        "pipeline_common": "통과",
        "text_match": "통과",
        "llm_refine": "통과(모의 API)",
        "실제 외부 API 호출": "수행하지 않음",
    },
    name="검증 결과",
)
display(verification_summary)
print("공통 유틸 모듈 노트북 검증 완료")

pipeline_common            통과
text_match                 통과
llm_refine         통과(모의 API)
실제 외부 API 호출          수행하지 않음
Name: 검증 결과, dtype: str

공통 유틸 모듈 노트북 검증 완료
